In [5]:
import numpy as np
import pandas as pd
import cartopy.crs as ccrs
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon, Point
from shapely.ops import transform
import matplotlib.colors as colors
import matplotlib.pyplot as plt
import xarray as xr
from glob import glob
from windspharm.xarray import VectorWind

In [6]:
# read in basin definition file
polygons_dict = {}

# read in basin definition file
with open("tc_basins.dat", "r") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue

        parts = line.split(",")
        basin_name = parts[0].replace('"', '')
        n_vertices = int(parts[1])

        lon_vals = list(map(float, parts[2:2+n_vertices]))
        lat_vals = list(map(float, parts[2+n_vertices:2+2*n_vertices]))

        coords = list(zip(lon_vals, lat_vals))
        poly = Polygon(coords)

        if basin_name not in polygons_dict:
            polygons_dict[basin_name] = []
        polygons_dict[basin_name].append(poly)

# Convert to GeoDataFrame
basin_records = []

for name, poly_list in polygons_dict.items():
    if len(poly_list) == 1:
        geom = poly_list[0]
    else:
        geom = MultiPolygon(poly_list)

    basin_records.append({
        "basin name": name,
        "geometry": geom
    })

basins = gpd.GeoDataFrame(basin_records, crs="EPSG:4326")

# fix invalid polygons
basins["geometry"] = basins["geometry"].buffer(0)

# remove empy geometries
basins = basins[~basins.geometry.is_empty]

# convert basins' lon to -180-180
basins["geometry"] = basins["geometry"].apply(
    lambda geom: transform(
        lambda x, y: (((x + 180) % 360) - 180, y),
        geom
    )
)

# read in NAtl subbasin polygons
sub_polygons_dict = {}

with open("tc_subbasins_NAtl_v5.dat", "r") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue

        parts = line.split(",")
        sub_basin_name = parts[0].replace('"', '')
        n_vertices = int(parts[1])

        lon_vals = list(map(float, parts[2:2+n_vertices]))
        lon_vals = [(lon + 180) % 360 - 180 for lon in lon_vals]
        lat_vals = list(map(float, parts[2+n_vertices:2+2*n_vertices]))

        coords = list(zip(lon_vals, lat_vals))
        poly = Polygon(coords)

        if sub_basin_name not in sub_polygons_dict:
            sub_polygons_dict[sub_basin_name] = []
        sub_polygons_dict[sub_basin_name].append(poly)

# Convert to GeoDataFrame
sub_basin_records = []

for name, poly_list in sub_polygons_dict.items():
    if len(poly_list) == 1:
        geom = poly_list[0]
    else:
        geom = MultiPolygon(poly_list)

    sub_basin_records.append({
        "sub_basin_name": name,
        "geometry": geom
    })

sub_basins = gpd.GeoDataFrame(sub_basin_records, crs="EPSG:4326",geometry="geometry")

# fix invalid polygons
sub_basins["geometry"] = sub_basins["geometry"].buffer(0)

# remove empty geometries
sub_basins = sub_basins[~sub_basins.geometry.is_empty]

# longitude conversion
import shapely.ops
def shift_lon(geom):
    return shapely.ops.transform(
        lambda x, y: (((x + 180) % 360) - 180, y),
        geom
    )

# shift lon
sub_basins["geometry"] = sub_basins["geometry"].apply(shift_lon)

In [ ]:
# load in u and v wind data to calculate 850hPa vorticity
ds1 = xr.open_mfdataset(
    "datasets/u-wind/post_processing/post_landmask/uwnd850_1940-2025_landmasked.nc", 
    combine = "by_coords", 
    preprocess=lambda ds: ds.drop_vars("time_bnds", errors="ignore")
)
ds2 = xr.open_mfdataset(
    "datasets/v-wind/post_processing/post_landmask/vwnd850_1940-2025_landmasked.nc", 
    combine = "by_coords", 
    preprocess=lambda ds: ds.drop_vars("time_bnds", errors="ignore")
)

# print(ds1)
# print(ds2)

# select the variables
uwnd = ds1["u"]
vwnd = ds2["v"]

# convert lon to -180-180
uwnd = uwnd.assign_coords(
    lon=(((uwnd.lon + 180) % 360) - 180)
).sortby("lon")
vwnd = vwnd.assign_coords(
    lon=(((vwnd.lon + 180) % 360) - 180)
).sortby("lon")

# # filter to relative humidity to a specific pressure level only
# uwnd850 = uwnd.sel(pressure=850)
# vwnd850 = vwnd.sel(pressure=850)

# # filter to hurricane season
# uwnd850_szn = (
#     uwnd850
#     .where(uwnd850.time.dt.month.isin([6, 7, 8, 9, 10]), drop=True)
# )
# vwnd850_szn = (
#     vwnd850
#     .where(vwnd850.time.dt.month.isin([6, 7, 8, 9, 10]), drop=True)
# )

# Ensure matching coordinates
uwnd, vwnd = xr.align(uwnd, vwnd, join="exact")

# Constants
a = 6.371e6
omega = 7.2921e-5

# Output directory
out_dir = "datasets/GPI/GPI_EN_calc/post_landmask"

# Process and save one year at a time
years = np.unique(uwnd.time.dt.year.values)
years = years[years >= 2002]

for year in years:
    print(f"Processing {year}...")

    # Select one year's data
    u = uwnd.sel(time=str(year))
    v = vwnd.sel(time=str(year))

    # Latitude in radians
    lat_rad = np.deg2rad(u["lat"])
    coslat = np.cos(lat_rad)

    # Relative vorticity
    dvdlam = v.differentiate("lon") * (180 / np.pi)

    ducos_dphi = (
        (u * coslat).differentiate("lat") * (180 / np.pi)
    )

    relative_vort = (
        dvdlam - ducos_dphi
    ) / (a * coslat)

    # Planetary vorticity
    f = 2 * omega * np.sin(lat_rad)

    # Absolute vorticity
    abs_vort = relative_vort + f

    abs_vort.name = "abs_vort850"
    abs_vort.attrs["long_name"] = "850 hPa absolute vorticity"
    abs_vort.attrs["units"] = "s-1"

    # Write this year's output to its own file
    out_path = f"{out_dir}/abs_vort_850_{year}.nc"

    abs_vort.to_dataset().to_netcdf(out_path)

    print(f"Saved {out_path}")

    # Release intermediate references
    del u, v, relative_vort, abs_vort, f, dvdlam, ducos_dphi

# Close input datasets
ds1.close()
ds2.close()



Processing 2002...
Saved datasets/GPI/GPI_EN_calc/post_landmask/abs_vort_850_2002.nc
Processing 2003...
Saved datasets/GPI/GPI_EN_calc/post_landmask/abs_vort_850_2003.nc
Processing 2004...
Saved datasets/GPI/GPI_EN_calc/post_landmask/abs_vort_850_2004.nc
Processing 2005...
Saved datasets/GPI/GPI_EN_calc/post_landmask/abs_vort_850_2005.nc
Processing 2006...
Saved datasets/GPI/GPI_EN_calc/post_landmask/abs_vort_850_2006.nc
Processing 2007...
Saved datasets/GPI/GPI_EN_calc/post_landmask/abs_vort_850_2007.nc
Processing 2008...
Saved datasets/GPI/GPI_EN_calc/post_landmask/abs_vort_850_2008.nc
Processing 2009...
Saved datasets/GPI/GPI_EN_calc/post_landmask/abs_vort_850_2009.nc
Processing 2010...
Saved datasets/GPI/GPI_EN_calc/post_landmask/abs_vort_850_2010.nc
Processing 2011...
Saved datasets/GPI/GPI_EN_calc/post_landmask/abs_vort_850_2011.nc
Processing 2012...
Saved datasets/GPI/GPI_EN_calc/post_landmask/abs_vort_850_2012.nc
Processing 2013...
Saved datasets/GPI/GPI_EN_calc/post_landmask/a

In [8]:
# potential intensity data
ds = xr.open_dataset("datasets/potential_intensity/pi_output_daily_landmasked.nc")

print(ds)

# select variable
vmax = ds['vmax']

print(vmax)

# save dataset
vmax.to_netcdf("datasets/GPI/GPI_EN_calc/potential_intensity_monthly_landmasked.nc")

<xarray.Dataset> Size: 19MB
Dimensions:      (time: 222, lat: 37, lon: 57)
Coordinates:
  * time         (time) datetime64[ns] 2kB 1981-09-01 1981-10-01 ... 2025-10-01
  * lat          (lat) float32 148B 90.0 87.5 85.0 82.5 80.0 ... 7.5 5.0 2.5 0.0
  * lon          (lon) float32 228B -120.0 -117.5 -115.0 ... 15.0 17.5 20.0
    spatial_ref  int64 8B ...
    number       int64 8B ...
    expver       <U4 16B ...
Data variables:
    vmax         (time, lat, lon) float64 4MB ...
    pmin         (time, lat, lon) float64 4MB ...
    ifl          (time, lat, lon) int64 4MB ...
    t0           (time, lat, lon) float64 4MB ...
    otl          (time, lat, lon) float64 4MB ...
<xarray.DataArray 'vmax' (time: 222, lat: 37, lon: 57)> Size: 4MB
[468198 values with dtype=float64]
Coordinates:
  * time         (time) datetime64[ns] 2kB 1981-09-01 1981-10-01 ... 2025-10-01
  * lat          (lat) float32 148B 90.0 87.5 85.0 82.5 80.0 ... 7.5 5.0 2.5 0.0
  * lon          (lon) float32 228B -120.0 -117